# Step 1: Dataset Curation, Validation, and Split Integrity

This notebook is the notebook-first entrypoint for SoundPulse dataset ingestion. It covers source discovery, metadata normalization, audio inspection, split generation, leakage checks, and export of a canonical manifest that downstream notebooks consume.

Expected output:
- A curated manifest JSON under `ml/data/curated/`
- A split summary and leakage report
- Inputs compatible with the shared SoundPulse contracts

In [ ]:
%pip install -q pandas numpy scikit-learn soundfile librosa matplotlib tqdm

In [ ]:
from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import librosa
import numpy as np
import pandas as pd
import soundfile as sf
from sklearn.model_selection import GroupShuffleSplit
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd().resolve().parents[1]
ML_ROOT = PROJECT_ROOT / 'ml'
CONTRACTS_DIR = PROJECT_ROOT / 'packages' / 'shared_contracts' / 'v1'
CURATED_DIR = ML_ROOT / 'data' / 'curated'
CURATED_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_OUTPUT_PATH = CURATED_DIR / 'soundpulse_manifest.v1.json'
SUMMARY_OUTPUT_PATH = CURATED_DIR / 'soundpulse_manifest_summary.csv'

with (CONTRACTS_DIR / 'class_labels.json').open('r', encoding='utf-8') as handle:
    CLASS_CONTRACT = json.load(handle)
with (CONTRACTS_DIR / 'preprocessing_config.json').open('r', encoding='utf-8') as handle:
    PREPROCESSING_CONTRACT = json.load(handle)

VALID_CLASS_SLUGS = {item['slug'] for item in CLASS_CONTRACT['classes']}
TARGET_SAMPLE_RATE = PREPROCESSING_CONTRACT['audio']['sample_rate_hz']
WINDOW_DURATION_MS = PREPROCESSING_CONTRACT['audio']['window_duration_ms']
VALID_SPLITS = ['train', 'validation', 'test']

print(f'Project root: {PROJECT_ROOT}')
print(f'Target sample rate: {TARGET_SAMPLE_RATE}')
print(f'Valid classes: {sorted(VALID_CLASS_SLUGS)}')

## Configure dataset sources

Update the metadata paths, audio roots, and label mappings below to match the copies of ESC-50, FSD50K, AudioSet subsets, and local Nigerian background recordings on your machine.

In [ ]:
DATASET_SOURCES: dict[str, dict[str, Any]] = {
    'esc50': {
        'source_name': 'ESC-50',
        'metadata_path': PROJECT_ROOT / 'external_data' / 'esc50' / 'meta' / 'esc50.csv',
        'audio_root': PROJECT_ROOT / 'external_data' / 'esc50' / 'audio',
        'filename_column': 'filename',
        'label_column': 'category',
        'fold_column': 'fold',
        'group_column': 'src_file',
        'label_map': {
            'fireworks': 'fire_alarm',
            'clock_alarm': 'clock_alarm',
            'crying_baby': 'crying_baby',
            'dog': 'dog_barking',
            'water_drops': 'water_running'
        }
    },
    'fsd50k_subset': {
        'source_name': 'FSD50K',
        'metadata_path': PROJECT_ROOT / 'external_data' / 'fsd50k_subset' / 'metadata.csv',
        'audio_root': PROJECT_ROOT / 'external_data' / 'fsd50k_subset' / 'audio',
        'filename_column': 'file_name',
        'label_column': 'soundpulse_label',
        'group_column': 'source_group_id',
        'label_map': {}
    },
    'audioset_subset': {
        'source_name': 'AudioSet',
        'metadata_path': PROJECT_ROOT / 'external_data' / 'audioset_subset' / 'metadata.csv',
        'audio_root': PROJECT_ROOT / 'external_data' / 'audioset_subset' / 'audio',
        'filename_column': 'file_name',
        'label_column': 'soundpulse_label',
        'group_column': 'youtube_id',
        'label_map': {}
    },
    'nigeria_background': {
        'source_name': 'Local ambient recordings',
        'metadata_path': PROJECT_ROOT / 'external_data' / 'nigeria_background' / 'metadata.csv',
        'audio_root': PROJECT_ROOT / 'external_data' / 'nigeria_background' / 'audio',
        'filename_column': 'file_name',
        'label_column': 'soundpulse_label',
        'group_column': 'session_id',
        'label_map': {}
    }
}

pd.DataFrame([
    {
        'dataset': name,
        'metadata_path': str(config['metadata_path']),
        'audio_root': str(config['audio_root'])
    }
    for name, config in DATASET_SOURCES.items()
])

In [ ]:
@dataclass
class CuratedClip:
    dataset_name: str
    source_name: str
    clip_id: str
    class_slug: str
    relative_path: str
    duration_ms: int
    sample_rate_hz: int
    source_group_id: str


def canonicalize_label(raw_label: str, label_map: dict[str, str]) -> str | None:
    label = str(raw_label).strip()
    label = label_map.get(label, label)
    return label if label in VALID_CLASS_SLUGS else None


def inspect_audio(audio_path: Path) -> tuple[int, int]:
    info = sf.info(str(audio_path))
    duration_ms = int(round((info.frames / info.samplerate) * 1000))
    return duration_ms, int(info.samplerate)


def build_curated_rows(dataset_name: str, config: dict[str, Any]) -> list[CuratedClip]:
    metadata_path = Path(config['metadata_path'])
    audio_root = Path(config['audio_root'])
    if not metadata_path.exists():
        print(f'Skipping {dataset_name}: metadata file not found -> {metadata_path}')
        return []

    frame = pd.read_csv(metadata_path)
    curated_rows: list[CuratedClip] = []
    for row in tqdm(frame.to_dict(orient='records'), desc=f'curating:{dataset_name}'):
        class_slug = canonicalize_label(row[config['label_column']], config.get('label_map', {}))
        if class_slug is None:
            continue

        audio_path = audio_root / str(row[config['filename_column']])
        if not audio_path.exists():
            continue

        duration_ms, sample_rate_hz = inspect_audio(audio_path)
        group_value = row.get(config.get('group_column', ''), row[config['filename_column']])
        clip_id = f"{dataset_name}-{Path(row[config['filename_column']]).stem}"
        relative_path = str(audio_path.relative_to(audio_root.parent))

        curated_rows.append(
            CuratedClip(
                dataset_name=dataset_name,
                source_name=config['source_name'],
                clip_id=clip_id,
                class_slug=class_slug,
                relative_path=relative_path,
                duration_ms=duration_ms,
                sample_rate_hz=sample_rate_hz,
                source_group_id=f"{dataset_name}-{group_value}",
            )
        )

    return curated_rows

In [ ]:
all_rows: list[CuratedClip] = []
for dataset_name, config in DATASET_SOURCES.items():
    all_rows.extend(build_curated_rows(dataset_name, config))

curated_df = pd.DataFrame([row.__dict__ for row in all_rows])
curated_df.head()

In [ ]:
assert not curated_df.empty, 'No rows were curated. Update DATASET_SOURCES to point at real metadata and audio.'
assert curated_df['clip_id'].is_unique, 'clip_id values must be unique'
assert set(curated_df['class_slug']).issubset(VALID_CLASS_SLUGS)

print(curated_df.groupby('class_slug').size().sort_values(ascending=False))

## Group-aware split generation

We split by `source_group_id` to avoid train, validation, and test leakage across derivative clips from the same source event.

In [ ]:
def assign_group_splits(frame: pd.DataFrame, seed: int = 42) -> pd.DataFrame:
    splitter_outer = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    train_val_idx, test_idx = next(splitter_outer.split(frame, groups=frame['source_group_id']))

    train_val = frame.iloc[train_val_idx].reset_index(drop=True)
    test = frame.iloc[test_idx].reset_index(drop=True)

    splitter_inner = GroupShuffleSplit(n_splits=1, test_size=0.125, random_state=seed)
    train_idx, validation_idx = next(splitter_inner.split(train_val, groups=train_val['source_group_id']))

    train = train_val.iloc[train_idx].assign(split='train')
    validation = train_val.iloc[validation_idx].assign(split='validation')
    test = test.assign(split='test')

    combined = pd.concat([train, validation, test], ignore_index=True)
    return combined


split_df = assign_group_splits(curated_df)
split_df.groupby(['split', 'class_slug']).size().unstack(fill_value=0)

In [ ]:
group_split_counts = split_df.groupby('source_group_id')['split'].nunique()
leakage_groups = group_split_counts[group_split_counts > 1]
assert leakage_groups.empty, f'Split leakage detected for groups: {list(leakage_groups.index)}'
assert set(split_df['split']).issubset(set(VALID_SPLITS))
assert (split_df['sample_rate_hz'] == TARGET_SAMPLE_RATE).all(), 'All clips must match the runtime sample rate contract'

split_summary = split_df.groupby(['split', 'class_slug']).size().reset_index(name='count')
split_summary

In [ ]:
manifest = {
    'version': '1.0.0',
    'description': 'Curated SoundPulse dataset manifest built from notebook step 1.',
    'classes_version': CLASS_CONTRACT['version'],
    'datasets': [],
}

for dataset_name, dataset_frame in split_df.groupby('dataset_name'):
    source_name = dataset_frame['source_name'].iloc[0]
    manifest['datasets'].append({
        'name': dataset_name,
        'source': source_name,
        'clips': dataset_frame[[
            'clip_id',
            'class_slug',
            'relative_path',
            'split',
            'duration_ms',
            'sample_rate_hz',
            'source_group_id',
        ]].to_dict(orient='records')
    })

with MANIFEST_OUTPUT_PATH.open('w', encoding='utf-8') as handle:
    json.dump(manifest, handle, indent=2)
split_summary.to_csv(SUMMARY_OUTPUT_PATH, index=False)

print(f'Wrote manifest -> {MANIFEST_OUTPUT_PATH}')
print(f'Wrote split summary -> {SUMMARY_OUTPUT_PATH}')

In [ ]:
from pathlib import Path
import sys

SRC_DIR = ML_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from soundpulse_ml.datasets import summarize_dataset_manifest

print(summarize_dataset_manifest(MANIFEST_OUTPUT_PATH))